In [ ]:
"""
Esercizio 1
concatenare due array NumPy di vendite giornaliere e transazioni di Dataframe, reset indice
"""

import numpy as np
import pandas as pd

# DATO SORGENTE ARRAY: data, prodotto, importo
vendite_arr1 = np.array([
    ['2026-10-08','Prodotto A', 150.50],
    ['2026-08-09','Prodotto B', 34.45]
])

vendite_arr2 = np.array([
    ['2026-10-08','Prodotto A', 150.50],
    ['2026-08-30','Prodotto C', 100.00]
])


transazioni_df = pd.DataFrame({
    'Data':['2026-09-11','2026-10-11'],
    'Prodotto':['Prodotto A', 'Prodotto C'],
    'Importo':[100.40, 150.60]
})


# Concatenazione di 2 array numpy per colonna
vendite_arr = np.concat([vendite_arr1, vendite_arr2], axis=0)

print(vendite_arr)

# Conversione e concatenazione 
vendite_df = pd.DataFrame(vendite_arr, columns= transazioni_df.columns)

vendite_finale_df = pd.concat([vendite_df, transazioni_df], ignore_index= True)
vendite_finale_df['Importo'] = vendite_finale_df['Importo'].astype('float').round(2)
vendite_finale_df['Data'] = pd.to_datetime(vendite_finale_df['Data'])

print("\nDATA FRAME CONCATENATO FINALE \n", vendite_finale_df)

In [ ]:
"""
Esercizio 2
normalizzare un dizionario annidato di transazioni per cliente e fare merge con dati anagrafici
"""

# DataFrame Anagrafica Clienti
anagrafica_clienti_df = pd.DataFrame({
    'ID_Cliente': ['C001', 'C002', 'C003', 'C004'],
    'Nome': ['Mario', 'Giulia', 'Luca', 'Elena'],
    'Cognome': ['Rossi', 'Bianchi', 'Verdi', 'Russo'],
    'Data_Nascita': pd.to_datetime(['1985-04-12', '1992-09-23', '1978-11-05', '1998-01-30']),
    'Genere': ['M', 'F', 'M', 'F'],
    'Citta': ['Milano', 'Roma', 'Torino', 'Napoli'],
    'Provincia': ['MI', 'RM', 'TO', 'NA'],
    'Cap': ['20121', '00118', '10121', '80121'],
    'Data_Iscrizione': pd.to_datetime(['2023-01-15', '2023-05-20', '2024-02-10', '2024-06-01']),
    'Segmento_Cliente': ['Premium', 'Standard', 'Standard', 'Premium'],
    'email':['mario.rossi@example.com','giulia.bianchi@example.com','elena.russo@example.com','elena.russo@example.com']
})

print("DATI ANAGRAFICI UTENTI \n",anagrafica_clienti_df)

# Dizionario transazioni annidato
dizionario_transazioni = {
    "TX_1001": {
        "data": "2026-10-01",
        "importo": 150.50,
        "metodo_pagamento": "Carta di Credito",
        "prodotti": ["Prodotto A", "Prodotto B"],
        "user": {
            "id_cliente": "C001",
            "nome": "Mario",
            "cognome": "Rossi",
            "email": "mario.rossi@example.com"
        },
        "account": {
            "type": "Premium"
        }
    },
    "TX_1002": {
        "data": "2026-10-02",
        "importo": 80.00,
        "metodo_pagamento": "PayPal",
        "prodotti": ["Prodotto B"],
        "user": {
            "id_cliente": "C002",
            "nome": "Giulia",
            "cognome": "Bianchi",
            "email": "giulia.bianchi@example.com"
        },
        "account": {
            "type": "Standard"
        }
    },
    "TX_1003": {
        "data": "2026-10-04",
        "importo": 210.00,
        "metodo_pagamento": "Bonifico",
        "prodotti": ["Prodotto A", "Prodotto D"],
        "user": {
            "id_cliente": "C004",
            "nome": "Elena",
            "cognome": "Russo",
            "email": "elena.russo@example.com"
        },
        "account": {
            "type": "Standard"
        }
    },
    "TX_1004": {
        "data": "2026-10-05",
        "importo": 45.00,
        "metodo_pagamento": "Carta di Credito",
        "prodotti": ["Prodotto C"],
        "user": {
            "id_cliente": "C001",  # Stesso cliente di TX_1001
            "nome": "Mario",
            "cognome": "Rossi",
            "email": "mario.rossi@example.com"
        },
        "account": {
            "type": "Premium"
        }
    }
}


transazioni = pd.json_normalize(list(dizionario_transazioni.values()))
transazioni.columns =[col.replace('user.','').replace('account.','') for col in transazioni.columns]

print("\nDIZIONARIO TRANSAZIONI ANNIDATO NORMALIZZATO \n", transazioni)

# Eseguo un merge tra transazioni e dati anagrafici dei clienti
df_merge = pd.merge(transazioni,anagrafica_clienti_df, left_on='id_cliente', right_on='ID_Cliente', how='left')

print("\nDATA FRAME FINALE CON MERGE TRA TRANSAZIONI E ANAGRAFICA CLIENTI\n")
df_merge.drop(columns=['email_y','id_cliente','nome','cognome'], inplace=True)
df_merge.rename(columns={'email_x':'email'}, inplace=True)
print(df_merge)

In [ ]:
"""
Esercizio 3
caricare JSON di promozioni da API e fare merge con vendite multiple, 
usando due chiavi e suffissi per colonne duplicate.

"""

import numpy as np
import pandas as pd
from pathlib import Path
import json

try:
    # Funziona se eseguito come file script (.py)
    cartella_script = Path(__file__).resolve().parent
except NameError:
    # Funziona se eseguito in Jupyter Notebook / Colab / Console REPL
    cartella_script = Path.cwd()


# Dati Json promozioni 
with open(cartella_script/'promozioni.json','r',encoding='utf-8') as f:
    dati_json = json.load(f) # 'dati_json' ora è una lista/dizionario Python in memoria

# trasforma i dati in dataframe
df_promozioni = pd.json_normalize(dati_json)

df_promozioni.columns = [col.replace('periodo.','') for col in df_promozioni.columns]
print("DATA FRAME PROMOZIONI CARICATO DA JSON\n", df_promozioni)


# Dati vendite multiple 
with open(cartella_script/'vendite_multiple.json','r',encoding='utf-8') as f:
    dati_json = json.load(f)

# trasforma i dati in dataframe
df_vendite_multiple = pd.json_normalize(
    dati_json,
    record_path=['prodotti'], #Prendi ogni singolo oggetto presente dentro la lista prodotti e creane una riga dedicata nel DataFrame
    meta=[ #Mentre crei le righe per i prodotti, duplica e mantieni accanto ad ogni prodotto anche queste informazioni di contesto generale
        'id_vendita',
        'data_ora',
        'stato',
        'promozione_applicata',
        ['cliente', 'id_cliente'], # Livello annidato: cliente -> id_cliente
        ['cliente', 'nome'],
        ['cliente', 'cognome'],
        ['pagamento', 'metodo'], # Livello annidato: pagamento -> metodo
        ['pagamento', 'importo_totale']
    ]
)

# pulizia nomi delle colonne esplose
df_vendite_multiple.columns = [
    col.replace("cliente.",''). 
    replace("prodotti.",''). 
    replace("pagamento.",'') for col in df_vendite_multiple.columns]
print("\nDATAFRAME VENDITE MULTIPLE CARICATO DA JSON\n", df_vendite_multiple)


# mergiamo i 2 dataframe di json 
df_vendite_promozioni = pd.merge(
    df_vendite_multiple, 
    df_promozioni, 
    left_on='promozione_applicata', 
    right_on='id_promozione',
    how='left', 
    suffixes=('_prodotto','_promozione')
)


print("\nDATAFRAME MERGIATO IN LEFT JOIN MANTENENDO LE RIGHE DI VENDITE MULTIPLE\n")
print(df_vendite_promozioni.head())

In [ ]:
"""
Esercizio 4
Molto difficile: integrare clienti, transazioni, promozioni e bonus in join multiplo con outer join, 
gestendo NaN, colonne duplicate e dati mancanti.

"""

import numpy as np 
import pandas as pd 
from pathlib import Path 
import json 

try:
    percorso_cartella = Path(__file__).resolve().parent
except NameError:
    percorso_cartella = Path.cwd()


# leggiamo i file json

# CLIENTI 
with open(percorso_cartella/'clienti_es4.json','r') as f:
    clienti_es4_data = json.load(f)

clienti_es4_df = pd.json_normalize(clienti_es4_data)

# PROMOZIONI 
with open(percorso_cartella/'promozioni_es4.json','r') as f:
    promozioni_es4_data = json.load(f)

promozioni_es4_df = pd.json_normalize(promozioni_es4_data)

# TRANSAZIONI 
with open(percorso_cartella/'transazioni_es4.json','r') as f:
    transazioni_es4_data = json.load(f)

transazioni_es4_df = pd.json_normalize(
    transazioni_es4_data,
    record_path=['prodotti'], # La chiave contenente la lista da esplodere
    meta=[
        'id_transazione',
        'id_cliente',
        'data',
        'promozione_applicata',
        'note'
    ] # I campi di livello superiore da ripetere per ogni prodotto
)

# BONUS 
with open(percorso_cartella/'bonus_es4.json','r') as f:
    bonus_es4_data = json.load(f)

bonus_es4_df = pd.json_normalize(bonus_es4_data)

# MERGIAMO I VARI DATAFRAME

merge_trans_clienti_df = pd.merge(transazioni_es4_df,clienti_es4_df,on='id_cliente',how='outer')

merge_trans_clienti_prom_df = pd.merge(
    merge_trans_clienti_df,
    promozioni_es4_df,
    left_on='promozione_applicata',
    right_on='id_promozione', 
    how='left',
    suffixes=('_trans','_cli')
)

merge_trans_clienti_prom_bonus_df = pd.merge(
    merge_trans_clienti_prom_df,
    bonus_es4_df,
    left_on=['id_transazione','id_cliente'],
    right_on=['id_transazione','id_cliente'],
    how='left',
    suffixes=('_trans','_bonus')
)


print("\nTUTTI I DATAFRAME MERGIATI\n")
print(merge_trans_clienti_prom_bonus_df.head())



# GESTIONE DEI VAORI MANCANTI O NONE
merge_trans_clienti_prom_bonus_df['promozione_applicata'] = merge_trans_clienti_prom_bonus_df['promozione_applicata'].fillna('NESSUNA PROMO')
merge_trans_clienti_prom_bonus_df['note'] = merge_trans_clienti_prom_bonus_df['note'].fillna('Nessuna nota')
merge_trans_clienti_prom_bonus_df['citta'] = merge_trans_clienti_prom_bonus_df['citta'].fillna('Nessuna citta')
merge_trans_clienti_prom_bonus_df['telefono'] = merge_trans_clienti_prom_bonus_df['telefono'].fillna('Nessun telefono')
merge_trans_clienti_prom_bonus_df['id_promozione'] = merge_trans_clienti_prom_bonus_df['id_promozione'].fillna('NESSUNA PROMO')
merge_trans_clienti_prom_bonus_df['descrizione'] = merge_trans_clienti_prom_bonus_df['descrizione'].fillna('Nessuna descrizione')
merge_trans_clienti_prom_bonus_df['id_bonus'] = merge_trans_clienti_prom_bonus_df['id_bonus'].fillna('Nessun id_bonus')
merge_trans_clienti_prom_bonus_df['tipo_bonus'] = merge_trans_clienti_prom_bonus_df['tipo_bonus'].fillna('Nessun tipo_bonus')


print("\nTUTTI I DATAFRAME MERGIATI CON GESTIONE VALORI TESTUALI MANCANTI\n")
print(merge_trans_clienti_prom_bonus_df.head())

# GESTIONE COLONNE NUMERICHE 
colonne_numeriche_zero = ['quantita','percentuale_sconto','sconto_fisso_eur','punti_esperienza','cashback_eur']
merge_trans_clienti_prom_bonus_df[colonne_numeriche_zero] = merge_trans_clienti_prom_bonus_df[colonne_numeriche_zero].fillna(0)
merge_trans_clienti_prom_bonus_df['prezzo_unitario'] = merge_trans_clienti_prom_bonus_df.groupby('sku')['prezzo_unitario'].transform(lambda x: x.fillna(x.median()))

print("\nTUTTI I DATAFRAME MERGIATI CON GESTIONE VALORI TESTUALI MANCANTI E NUMERICI\n")
print(merge_trans_clienti_prom_bonus_df.head())

merge_trans_clienti_prom_bonus_df.to_csv(percorso_cartella/'dataframe_mergiato.csv',index=False,sep=';',encoding='utf-8')


TUTTI I DATAFRAME MERGIATI

      sku       nome_prodotto  prezzo_unitario  quantita id_transazione  \
0  PRD_10   Tastiera Wireless             45.0         1    TX_2026_001   
1  PRD_12        Mouse Ottico             20.0         2    TX_2026_001   
2  PRD_05        Cavo HDMI 2m             12.5         3    TX_2026_003   
3  PRD_99  Monitor 27 Pollici            250.0         1    TX_2026_002   
4  PRD_20    Cuffie Bluetooth             80.0         1    TX_2026_004   

  id_cliente        data promozione_applicata  \
0     CLI_01  2026-10-01      SPEDIZIONE_FREE   
1     CLI_01  2026-10-01      SPEDIZIONE_FREE   
2     CLI_01  2026-10-05                 None   
3     CLI_02  2026-10-03   PROMO_AUTUNNO_2026   
4     CLI_03  2026-10-06       PROMO_INVALIDA   

                                note           nome  ...   citta  \
0              Consegnare di mattina    Laura Rossi  ...  Milano   
1              Consegnare di mattina    Laura Rossi  ...  Milano   
2                    